# CALM-VAD — multi-dataset evaluation (Colab)

Runs the full CALM-VAD pipeline (M1 reliability → M2 evidence fusion → M3
calibration → M4 alarm budget) and the deployable evaluation harness across
several pose-VAD benchmarks, using **pre-extracted skeletons + ground truth**
published by the community — **no video downloads, no GPU pose extraction**.

| dataset | source | format |
|---|---|---|
| ShanghaiTech | STG-NF release | GEPC json |
| HR-ShanghaiTech / HR-Avenue / HR-UBnormal | MoCoDAD release | Morais csv |

Run cells top-to-bottom. Cell 4 loops over whichever datasets you pick.
A GPU is **not** needed; a CPU runtime is fine.

## Cell 1 — install + code

In [ ]:
!pip -q install numpy scipy scikit-learn pyyaml gdown
import os, sys, subprocess
sh = lambda cmd: subprocess.run(cmd, shell=True, check=False)

REPO_URL = 'https://github.com/FaizanAbbas512/Sentrix.git'   # '' to upload a zip instead
if REPO_URL:
    sh('rm -rf /content/sentrix && git clone --depth 1 %s /content/sentrix' % REPO_URL)
else:
    from google.colab import files
    up = files.upload(); z = next(iter(up))
    sh('rm -rf /content/sentrix && mkdir -p /content/sentrix && unzip -q "%s" -d /content/sentrix' % z)
    subs = [d for d in os.listdir('/content/sentrix') if os.path.isdir(f'/content/sentrix/{d}')]
    if 'calm' not in subs and len(subs) == 1:
        sh(f'cp -r /content/sentrix/{subs[0]}/* /content/sentrix/')

os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
assert os.path.isdir('calm'), 'calm/ not found - check REPO_URL or the uploaded zip'
for d in ('data/pose', 'data/raw', 'results'): os.makedirs(d, exist_ok=True)
sh('python -m calm.selftest | tail -3')

## Cell 2 — dataset registry

Two ways to get data:
* **`gdrive_file`** — a *single* Drive zip. `gdown` fetches it directly.
  (STG-NF publishes ShanghaiTech + UBnormal this way — works out of the box.)
* **`drive_zip`** — for the MoCoDAD HR poses. `gdown` **cannot** pull a Drive
  *folder* with >50 files, so you download it once by hand:
    1. open <https://drive.google.com/drive/folders/1aUDiyi2FCc6nKTNuhMvpGG_zLZzMMc83>
    2. right-click the dataset folder (**Avenue** / **ShanghaiTech** / **UBnormal**)
       → **Download** → Google zips it
    3. put that zip in **your Google Drive** (any folder) *or* keep it to upload.
  Cell 3 finds it by name (mounts your Drive, or asks you to upload).

Edit `RUN` to pick datasets. `verify the ids on the repo READMEs if a fetch fails.`

In [ ]:
DATASETS = {
  # single-file Drive zips (gdown works):
  'shanghaitech': dict(gdrive_file='1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt', fps=24),
  'ubnormal':     dict(gdrive_file='1o9h3Kh6zovW4FIHpNBGnYIRSbGCu-qPt', fps=30),  # same STG-NF zip
  # MoCoDAD HR poses — download the Drive folder as a zip by hand (see Cell 2 text):
  'hr_avenue':      dict(drive_zip='*venue*.zip',        fps=25),
  'hr_shanghaitech':dict(drive_zip='*hanghai*.zip',      fps=24),
  'hr_ubnormal':    dict(drive_zip='*bnormal*.zip',      fps=30),
}

RUN = ['shanghaitech']          # <- start with this; add 'hr_avenue' etc once its zip is ready

# custom link?  DATASETS['shanghaitech']['gdrive_file'] = 'NEW_ID_OR_LINK'

## Cell 3 — helpers (download / find zip / extract)

In [ ]:
import os, glob, zipfile, tarfile, gdown, fnmatch
os.makedirs('data/raw', exist_ok=True)
_DRIVE_MOUNTED = [False]

def _extract(arc, dest):
    os.makedirs(dest, exist_ok=True)
    if zipfile.is_zipfile(arc):     zipfile.ZipFile(arc).extractall(dest)
    elif tarfile.is_tarfile(arc):   tarfile.open(arc).extractall(dest)
    else: raise RuntimeError(f'{arc} is not a zip/tar')
    # extract any nested archives once
    for p in glob.glob(f'{dest}/**/*', recursive=True):
        if p != arc and (zipfile.is_zipfile(p) or (p.endswith(('.tar','.tgz','.tar.gz')) and tarfile.is_tarfile(p))):
            try: _extract(p, os.path.dirname(p))
            except Exception: pass

def _find_zip(pattern):
    # look in /content, then the user's Drive
    for base in ('/content', '/content/drive/MyDrive'):
        if base.startswith('/content/drive') and not _DRIVE_MOUNTED[0]:
            try:
                from google.colab import drive; drive.mount('/content/drive'); _DRIVE_MOUNTED[0] = True
            except Exception: continue
        for p in glob.glob(f'{base}/**/*', recursive=True):
            if fnmatch.fnmatch(os.path.basename(p).lower(), pattern.lower()) and p.lower().endswith(('.zip','.tar','.tgz','.tar.gz')):
                return p
    return None

def fetch(name, spec):
    dest = f'data/raw/{name}'
    if os.path.isdir(dest) and glob.glob(f'{dest}/**/*.npy', recursive=True):
        return dest                                    # already have it
    if 'gdrive_file' in spec:
        g = spec['gdrive_file']
        src = g if 'http' in g else f'https://drive.google.com/uc?id={g}'
        gdown.download(src, f'data/raw/{name}.zip', quiet=False, fuzzy=True)
        _extract(f'data/raw/{name}.zip', dest)
    else:                                              # drive_zip : find it or ask to upload
        z = _find_zip(spec['drive_zip'])
        if not z:
            from google.colab import files
            print(f"no zip matching {spec['drive_zip']} in /content or Drive — upload it now:")
            up = files.upload(); z = os.path.join('/content', next(iter(up)))
        print('  using zip:', z)
        _extract(z, dest)
    return dest
print('helpers ready')

## Cell 4 — run every dataset in `RUN`

Each one: fetch → `calm.datasets.inspect` (shows the tree) → full harness via
`--auto` → `results/calm_report_<name>.{json,txt}` + reusable `data/pose/<name>.json`.

In [ ]:
import os, subprocess
done = []
for name in RUN:
    spec = DATASETS[name]
    print('\n' + '=' * 62 + '\n  ' + name + '\n' + '=' * 62)
    try:
        root = fetch(name, spec)
    except Exception as e:
        print('  fetch failed:', e); continue
    subprocess.run(['python', '-m', 'calm.datasets', root])          # show the tree
    rc = subprocess.run(['python', '-m', 'calm.harness', '--auto', root,
                         '--tag', name, '--fps', str(spec['fps']),
                         '--save-generic', f'data/pose/{name}.json']).returncode
    if rc == 0 and os.path.exists(f'results/calm_report_{name}.json'):
        done.append(name); print('  OK ->', f'results/calm_report_{name}.txt')
    else:
        print('  !! harness failed for', name, '— read the tree / errors above')
print('\nfinished:', done)

## Cell 5 — combined comparison table (all datasets, CALM-VAD vs baselines)

In [ ]:
import json, glob, os
rows = []
for jp in sorted(glob.glob('results/calm_report_*.json')):
    r = json.load(open(jp)); ds = r['tag']
    cal = r['calibration']
    for s in r['streams']:
        rows.append(dict(dataset=ds, method=s['label'],
                         AUC=round(s['frame_auc'], 3),
                         eventF1=round(s['event']['f1_avg'], 3),
                         **{f"FAPH@{k.split('=')[-1]}": s[k] for k in s if k.startswith('faph@')}))
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    from IPython.display import display; display(df)
except Exception:
    for x in rows: print(x)

print('\nCalibration (CALM-VAD, ECE raw -> cal) and cost per dataset:')
for jp in sorted(glob.glob('results/calm_report_*.json')):
    r = json.load(open(jp)); c = r['calibration']
    print(f"  {r['tag']:<16} ECE {c['ece_raw']:.4f} -> {c['ece_cal']:.4f}   "
          f"decision {r['cost']['decision_layer_ms_per_frame_mean']:.2f} ms/frame")

## Cell 6 — cross-dataset generalisation (fit on A, test on B)

Uses the `data/pose/<name>.json` files saved in Cell 4.

In [ ]:
import json, itertools, os
PAIRS = list(itertools.permutations([n for n in RUN if os.path.exists(f'data/pose/{n}.json')], 2))
for a, b in PAIRS:
    A = json.load(open(f'data/pose/{a}.json')); B = json.load(open(f'data/pose/{b}.json'))
    for x in A['clips']: x['split'] = 'calib'
    for x in B['clips']: x['split'] = 'test'
    out = f'data/pose/{a}__to__{b}.json'
    json.dump({'fps': A['fps'], 'clips': A['clips'] + B['clips']}, open(out, 'w'))
    os.system(f"python -m calm.harness --generic {out} --tag {a}__to__{b}")
    r = json.load(open(f'results/calm_report_{a}__to__{b}.json'))
    s = [x for x in r['streams'] if 'CALM' in x['label']][0]
    print(f'  {a} -> {b}:  eventF1 {s["event"]["f1_avg"]:.3f}  AUC {s["frame_auc"]:.3f}')

## Cell 7 — download all results + reusable pose files

In [ ]:
!zip -qr /content/calm_all.zip results data/pose/*.json
from google.colab import files; files.download('/content/calm_all.zip')

## Cell 8 (appendix) — GPU path: raw videos → poses

Only if you have raw video files + per-clip GT and want to extract poses
yourself (needs `Runtime → T4 GPU`). Otherwise ignore this cell.

In [ ]:
# !pip -q install ultralytics opencv-python-headless
# TAG = 'myset'
# !python -m calm.extract_poses --videos /content/vids --gt /content/gt \
#     --out data/pose/$TAG.json --split test --weights yolo11n-pose.pt --imgsz 640 --device 0
# !python -m calm.harness --generic data/pose/$TAG.json --tag $TAG